# Focused decoding stability: what if the predicted coefficients are slightly wrong?

This notebook uses **recut/hard only**. It asks whether coefficients between valid boundary variants, or coefficients with controlled opening errors, still produce sensible complete crypt meshes. No encoder or latent model is trained. Geometry is generated directly as `XYZ = basis_matrix @ coefficients`; there is no inverse parameterization or coefficient refitting during decoding.

Two real crypts are inspected: an ordinary example from **20251201**, selected near the previous pilot's median core drift, and the most difficult example from **20250929**. Both come from the existing day ≥3.5 pilot. We evaluate physical-size and unit-RMS representations separately, using the **saved, frozen cohort alignment references**.

Run notebooks 02 and 03 first to create the selection, basis, alignment references and boundary-study report. Use the `organoid-mesh` environment. This notebook reruns its fits and checks; it does not silently reuse result caches. Reports and figures go into `exports/crypt_morphology/decoding_stability/`.

**Read the small summary table and mesh sequences first.** Detailed numerical diagnostics are saved for inspection. A completed numerical calculation is not automatically an acceptable anatomical reconstruction.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/organograph").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio
from IPython.display import display
pio.renderers.default = "plotly_mimetype"
from organograph.crypt_morphology import *
from organograph.crypt_morphology.datasets import discover_hks_crypts, load_hks_crypt, load_hks_parent
from organograph.plotting.crypt_morphology import (
    plot_decoding_sequence, plot_decoding_sequence_static, plot_decoding_response,
)
ARTIFACTS = ROOT / "exports/crypt_morphology"
OUT = ARTIFACTS / "decoding_stability"
OUT.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path(os.environ.get("ORGANOGRAPH_DATA_ROOT", ROOT.parent / "NicoleData"))
SIZE_MODES = ("physical", "unit_rms")
GRID = (24, 96)
FINE_GRID = (48, 192)
RIM_MODES = (3, 12)
ERROR_MULTIPLIERS = (.25, .5, 1.)
INTERPOLATION_FRACTIONS = (.25, .5, .75)
PLOT_PATHS = ("uniform",)  # Also add "sideways" to display that tested path.
PLOT_RIM_MODES = (12,)  # Add 3 to display the smooth-error mesh sequence as well.
PLOT_SIZE_MODE = "physical"  # Change to unit_rms and rerun the plotting cells.
basis = load_basis(ARTIFACTS / "pilot_basis.json")
references = {mode: load_alignment_reference(ARTIFACTS / f"pilot_alignment_{mode}.npz", basis) for mode in SIZE_MODES}
selection = json.loads((ARTIFACTS / "selection.json").read_text())
def key_of(record):
    return (str(record["dataset"]), record["label_uid"], int(record["crypt_index"]))
# Resolve the saved logical selection against DATA_ROOT, including relocated data.
available = {key_of(r): r for r in discover_hks_crypts(DATA_ROOT)}
selected = [available[key_of(r)] for r in selection]
previous = pd.read_csv(ARTIFACTS / "boundary_stability/real_stability.csv", dtype={"dataset": str})
previous = previous[(previous.method == "recut_hard") & (previous.status == "ok") & (previous.variant != "baseline")]
ranked = previous.groupby(["dataset", "label_uid", "crypt_index"]).agg(
    median_drift=("core_drift_relative", "median"), worst_drift=("core_drift_relative", "max"), count=("variant", "count"))
target = ranked.median_drift.median()
ordinary = ranked[(ranked.index.get_level_values("dataset") == "20251201") & (ranked["count"] == 16)]
ordinary_key = (ordinary.median_drift-target).abs().idxmin()
difficult_key = ranked.worst_drift.idxmax()
case_keys = {"Ordinary crypt": ordinary_key, "Difficult crypt": difficult_key}
print("Common basis:", basis.max_degree, "degrees;", basis.n_modes, "XYZ modes")
display(pd.DataFrame([dict(case=case, dataset=key[0], label_uid=key[1], crypt_index=key[2])
                      for case, key in case_keys.items()]))

## Prepare reliable endpoint representations

Every selected opening is mapped onto the whole cap and fitted with area weights, without a padded region or blurred fitting weights. The 17 openings are the same baseline, offsets, and asymmetric perturbations used previously. Rejected cuts remain explicit.

All representations are aligned to the existing cohort reference for their size convention. We do **not** align each interpolated or perturbed output again: doing so could conceal deformation. In unit-RMS mode, each observed boundary variant uses its own measured radius, as it would in a shape-only dataset. Generated outputs are not renormalized after interpolation or perturbation.

The other 18 pilot crypts are fitted only at their original openings, to provide a limited comparison between boundary variation and variation across crypts. This is an input-geometry comparison, not a latent-space test or a biological classification benchmark.

In [ ]:
source = {}
for i, rec in enumerate(selected):
    patch, metadata = load_hks_crypt(rec)
    mapping = parameterize_patch(patch, half_angle=basis.half_angle)
    rep = fit_representation(patch, mapping, basis, metadata=metadata)
    source[key_of(rec)] = (patch, mapping, rep, metadata)
    print(f"Baseline {i+1}/{len(selected)}: {rec['label_uid']}")
all_keys = list(source)
aligned_baselines = {}
for mode in SIZE_MODES:
    reps = [with_size_mode(source[key][2], mode) for key in all_keys]
    aligned = align_to_reference(reps, basis, references[mode])
    aligned_baselines[mode] = dict(zip(all_keys, aligned.representations))

cases, fit_rows = {}, []
for case, key in case_keys.items():
    rec = next(r for r in selected if key_of(r) == key)
    v, f, ids, neighbors, metadata = load_hks_parent(rec)
    patch, mapping, raw_baseline, _ = source[key]
    radius = raw_baseline.rms_radius
    family = prepare_boundary_family(v, f, ids, max_displacement=.1*radius,
                                     collar_width=.3*radius, forbidden_vertex_ids=neighbors)
    raw = {"baseline": raw_baseline}
    for shift in opening_shifts(radius):
        if shift.name == "baseline":
            fit_rows.append(dict(case=case, variant=shift.name, status="ok", error=""))
            continue
        try:
            cut = family.cut(shift).patch
            mapped = parameterize_patch(cut, half_angle=basis.half_angle)
            raw[shift.name] = fit_representation(cut, mapped, basis)
            fit_rows.append(dict(case=case, variant=shift.name, status="ok", error=""))
        except ValueError as error:
            fit_rows.append(dict(case=case, variant=shift.name, status="rejected_or_failed", error=str(error)))
    core_mask = family.signed_distance[f[patch.source_face_ids]].max(axis=1) < -.1*radius
    by_mode = {}
    for mode in SIZE_MODES:
        variants = [name for name in raw if name != "baseline"]
        aligned = align_to_reference([with_size_mode(raw[name], mode) for name in variants], basis, references[mode])
        by_mode[mode] = dict(zip(variants, aligned.representations))
        by_mode[mode]["baseline"] = aligned_baselines[mode][key]
    cases[case] = dict(key=key, patch=patch, mapping=mapping, raw=raw, aligned=by_mode,
                       core_mask=core_mask, radius=radius, metadata=metadata,
                       boundary_diagnostics=family.diagnostics)
fit_table = pd.DataFrame(fit_rows)
fit_table.to_csv(OUT / "opening_fits.csv", index=False)
display(fit_table.groupby(["case", "status"]).size().rename("openings").to_frame())
if (fit_table.status != "ok").any():
    display(fit_table[fit_table.status != "ok"])
separation_rows = []
for case, data in cases.items():
    for mode in SIZE_MODES:
        ref = references[mode]
        q, _ = basis.reference_mesh(ref.n_rings, ref.n_sectors)
        baseline = data["aligned"][mode]["baseline"]
        distances = {key: coefficient_surface_distance(baseline.coefficients, rep.coefficients, basis, q, ref.weights)
                     for key, rep in aligned_baselines[mode].items() if key != data["key"]}
        nearest = min(distances, key=distances.get)
        within = [coefficient_surface_distance(baseline.coefficients, rep.coefficients, basis, q, ref.weights)
                  for name, rep in data["aligned"][mode].items() if name != "baseline"]
        separation_rows.append(dict(case=case, size_mode=mode, nearest_crypt=str(nearest),
            nearest_other_distance=distances[nearest], within_median=float(np.median(within)), within_max=float(np.max(within)),
            median_within_to_nearest_other=float(np.median(within)/distances[nearest]),
            max_within_to_nearest_other=float(np.max(within)/distances[nearest])))
separation = pd.DataFrame(separation_rows)
separation.to_csv(OUT / "within_between.csv", index=False)

## Simulate decoder errors

Two experiments generate coefficient arrays without pretending they are observed source meshes:

1. **Interpolation:** decode the 25%, 50%, and 75% mixtures between inward/outward cuts, and between opposite sideways cuts. Endpoints are independently fitted and aligned observations. Interpolation tests one plausible kind of compromise prediction; it is not a latent-space interpolation.
2. **Opening-only errors:** change only the rim coefficients, leaving the interior coefficients fixed. Use two deterministic directions: smooth rim modes 1–3, and finer modes 1–12, in both signs. The error sizes are 25%, 50%, and 100% of the median observed rim displacement across that crypt's physical-size boundary variants. This calibrates amplitude to the experiment, rather than adding arbitrary coefficient noise.

Rim RMS uses uniform cap angle, not physical rim arclength. The **same physical perturbations** are transformed into both size/alignment conventions. A small opening error can affect derivatives or fold a surface even when point displacements are small; finer modes test this explicitly.

The core consists of original triangles deeper than the maximum inward shift. Its evaluation points are original triangle **centroids**, distinct from the three points used in fitting. Two measurements are kept separate: motion of baseline-corresponding cap samples, and the distance from actual baseline core membrane points to the generated mesh. The latter is the main geometric-error measure below.

In [ ]:
probes, coefficients, scenario_info, rows, skipped = {}, {}, {}, [], []
paths = {"uniform": ("offset_-0.100", "offset_+0.100"),
         "sideways": ("tilt_0.100_0", "tilt_0.100_2")}
calibration_rows = []
for case, data in cases.items():
    physical = data["aligned"]["physical"]
    phi = np.arange(1024)*2*np.pi/1024
    rim_matrix = basis.evaluate(np.column_stack([np.full(len(phi), basis.half_angle), phi]))
    observed_rim = [np.sqrt(np.mean(np.sum((rim_matrix @ (rep.coefficients-physical["baseline"].coefficients))**2, axis=1)))
                    for name, rep in physical.items() if name != "baseline"]
    calibration = float(np.median(observed_rim)/data["radius"])
    calibration_rows.append(dict(case=case, original_rms_radius=data["radius"],
                                 calibrated_rim_rms_fraction=calibration))
    for mode in SIZE_MODES:
        fitted = data["aligned"][mode]
        baseline = fitted["baseline"]
        probe = DecodingProbe(data["patch"], data["mapping"], baseline, basis, data["core_mask"],
                              n_rings=GRID[0], n_sectors=GRID[1])
        probes[case, mode] = probe
        scenarios = [(name, rep.coefficients, dict(kind="observed_variant", variant=name,
                       alignment_phase=rep.phase)) for name, rep in fitted.items()]
        for path, (left, right) in paths.items():
            if left not in fitted or right not in fitted:
                skipped.append(dict(case=case, size_mode=mode, path=path, reason="A required endpoint was rejected"))
                continue
            for fraction in INTERPOLATION_FRACTIONS:
                name = f"blend_{path}_{fraction:.2f}"
                c = interpolate_coefficients(fitted[left].coefficients, fitted[right].coefficients, fraction, basis)
                scenarios.append((name, c, dict(kind="interpolation", path=path, fraction=fraction,
                                                endpoint_a=left, endpoint_b=right)))
        for max_mode in RIM_MODES:
            for multiplier in ERROR_MULTIPLIERS:
                amplitude = multiplier*calibration
                raw_delta = rim_coefficient_error(basis, amplitude*data["radius"], max_mode=max_mode, seed=2026+max_mode)
                delta = transport_coefficient_error(raw_delta, basis, data["raw"]["baseline"], baseline)
                for sign in [-1, 1]:
                    name = f"rim_m{max_mode}_x{multiplier:.2f}_{sign:+d}"
                    scenarios.append((name, baseline.coefficients+sign*delta, dict(kind="rim_error", max_mode=max_mode,
                        multiplier=multiplier, sign=sign, amplitude_fraction=amplitude, seed=2026+max_mode)))
        for name, c, info in scenarios:
            key = (case, mode, name)
            coefficients[key], scenario_info[key] = c, info
            measurements, _ = probe.evaluate(c)
            rows.append(dict(case=case, size_mode=mode, scenario=name, **info, **measurements))
        print(f"{case}, {mode}: {len(scenarios)} decoded meshes checked")
results = pd.DataFrame(rows)
results.to_csv(OUT / "decoding_checks.csv", index=False)
calibration = pd.DataFrame(calibration_rows)
calibration.to_csv(OUT / "rim_error_calibration.csv", index=False)
if skipped:
    display(pd.DataFrame(skipped))
print("Error amplitude calibration (percent of original RMS radius):")
display(calibration.assign(calibrated_rim_rms_percent=100*calibration.calibrated_rim_rms_fraction)
        [["case", "original_rms_radius", "calibrated_rim_rms_percent"]])
# Retain experiment coefficient arrays, not a training-dataset exporter.
keys = list(coefficients)
np.savez_compressed(OUT / "decoded_test_coefficients.npz", coefficients=np.stack([coefficients[k] for k in keys]),
                    case=np.array([k[0] for k in keys]), size_mode=np.array([k[1] for k in keys]),
                    scenario=np.array([k[2] for k in keys]), basis_id=basis.identifier)

## Check whether mesh resolution changes the verdict

Every output above is checked at 24 rings × 96 sectors. Baselines, the two uniform endpoints, interpolation midpoints, all largest opening errors, the largest core-error case, and any flagged outputs are also checked at **48 × 192**. This is a resolution control; it does not certify the continuous expansion or untested coefficient directions.

Intersection checks use bounding boxes and triangle separating axes, including coplanar overlap. Ordinary shared edges/vertices are excluded by contracting incident triangles toward their centroids by 1e−7; intersections confined to that tiny removed strip can be missed. Tests include known crossings, non-crossings, coplanar overlap, shared-corner contacts, and an independent convex-feasibility check. Numerical tolerance is relative to each triangle pair's extent.

A geometric flag means an intersection, near-zero-area triangle, or a >90° face-normal turn relative to the baseline. A normal turn is an inspection warning, not by itself a proof of a fold. Continuous stretch/error values are reported without declaring an arbitrary biological acceptance cutoff.

In [ ]:
fine_rows = []
for case, data in cases.items():
    for mode in SIZE_MODES:
        group = results[(results.case == case) & (results.size_mode == mode)]
        names = {"baseline", "offset_-0.100", "offset_+0.100", "blend_uniform_0.50", "blend_sideways_0.50"}
        names.update(group[(group.kind == "rim_error") & (group.multiplier == 1)].scenario)
        names.update(group[group.geometry_warning].scenario)
        names.add(group.loc[group.core_surface_rmse_relative.idxmax(), "scenario"])
        probe = DecodingProbe(data["patch"], data["mapping"], data["aligned"][mode]["baseline"], basis,
                              data["core_mask"], n_rings=FINE_GRID[0], n_sectors=FINE_GRID[1])
        for name in sorted(names):
            key = (case, mode, name)
            if key not in coefficients:
                continue
            measurements, _ = probe.evaluate(coefficients[key])
            fine_rows.append(dict(case=case, size_mode=mode, scenario=name, **scenario_info[key], **measurements))
        print(f"Fine checks completed: {case}, {mode}")
fine = pd.DataFrame(fine_rows)
fine.to_csv(OUT / "fine_mesh_checks.csv", index=False)
summary_rows = []
for (case, mode), group in results.groupby(["case", "size_mode"], sort=False):
    baseline = group[group.scenario == "baseline"].iloc[0]
    dense = fine[(fine.case == case) & (fine.size_mode == mode)]
    summary_rows.append({"Crypt": case, "Size mode": mode,
        "Baseline core error (%)": 100*baseline.core_surface_rmse_relative,
        "Worst interpolated core error (%)": 100*group[group.kind == "interpolation"].core_surface_rmse_relative.max(),
        "Worst rim-error core error (%)": 100*group[group.kind == "rim_error"].core_surface_rmse_relative.max(),
        "Flagged meshes / checked": f"{group.geometry_warning.sum()} / {len(group)}",
        "Flagged fine meshes / checked": f"{dense.geometry_warning.sum()} / {len(dense)}"})
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / "summary.csv", index=False)
display(summary.round(3))
print("Core errors are point-to-surface distances divided by the original RMS radius, not latent-code errors.")
flagged = pd.concat([results.assign(grid="standard"), fine.assign(grid="fine")], ignore_index=True)
flagged = flagged[flagged.geometry_warning]
if flagged.empty:
    print("No intersections, collapsed triangles, or >90-degree normal turns detected in these tests.")
else:
    print("Inspect these warnings before treating the corresponding predictions as usable:")
    display(flagged[["case", "size_mode", "scenario", "grid", "intersection_pairs", "collapsed_faces", "normal_turn_faces"]])
print("Input-space boundary variation relative to the nearest other pilot crypt (smaller ratios mean better separation):")
display(separation[["case", "size_mode", "median_within_to_nearest_other", "max_within_to_nearest_other", "nearest_crypt"]].round(3))

## Separate source-shape accuracy from alignment changes

The main summary measures generated geometry in the **fixed cohort coordinate frame**. A large core error can therefore include changes in alignment or size normalization, as well as deformation. Independently aligning each cut to the same template does not guarantee that its orientation stays near the baseline orientation.

For fitted observations only, we know the original source transform. The following control restores each observed variant to the same parent membrane coordinates, then applies the baseline transform and size convention. It measures fitting accuracy with alignment/normalization differences removed. This control is unavailable for arbitrary generated coefficients; it is **not** used to alter interpolation outputs or make the plotted meshes look better.

Compare this table with the main summary before interpreting large errors as decoder instability. We cannot infer that every interpolation error is rigid motion: interpolating coefficients after different pose/phase choices can itself deform a surface.

In [ ]:
source_control_rows = []
for case, data in cases.items():
    for mode in SIZE_MODES:
        probe = probes[case, mode]
        for name, rep in data["aligned"][mode].items():
            source_control_rows.append(dict(case=case, size_mode=mode, variant=name, **probe.source_frame_control(rep)))
source_controls = pd.DataFrame(source_control_rows)
source_controls.to_csv(OUT / "source_pose_control.csv", index=False)
control_summary = source_controls.groupby(["case", "size_mode"]).agg(
    worst_source_frame_core_error=("core_source_frame_rmse_relative", "max"),
    largest_source_rotation_change=("source_pose_rotation_degrees", "max"))
control_summary["worst_source_frame_core_error"] *= 100
control_summary = control_summary.rename(columns={"worst_source_frame_core_error": "Worst source-frame core error (%)",
    "largest_source_rotation_change": "Largest observed pose change (degrees)"})
display(control_summary.round(3))
print("These source-frame controls apply to fitted observations only, not the interpolated meshes.")

## Inspect the interpolation meshes

Blue surfaces are decoded directly from the endpoint or interpolated coefficients. Gray is the same frozen baseline in every panel; black is the decoded opening. All panels within a figure use identical spatial limits. No output is realigned or rescaled to improve its appearance.

The first and last panels are fitted observations of different cuts. The three middle panels are synthetic coefficient predictions. They have no uniquely defined true opening; the question is whether the shape changes smoothly without damaging the interior. The plots default to physical size; change `PLOT_SIZE_MODE` above to inspect unit-RMS results.

In [ ]:
for case in cases:
    probe = probes[case, PLOT_SIZE_MODE]
    for path, (left, right) in paths.items():
        if path not in PLOT_PATHS:
            continue
        names = [left, f"blend_{path}_0.25", f"blend_{path}_0.50", f"blend_{path}_0.75", right]
        if not all((case, PLOT_SIZE_MODE, name) in coefficients for name in names):
            continue
        vertices = [probe.matrix @ coefficients[case, PLOT_SIZE_MODE, name] for name in names]
        labels = ["Endpoint A", "25% toward B", "50% toward B", "75% toward B", "Endpoint B"]
        group = results[(results.case == case) & (results.size_mode == PLOT_SIZE_MODE)].set_index("scenario")
        title = f"{case}: {path} boundary variants — {PLOT_SIZE_MODE}"
        fig = plot_decoding_sequence(vertices, probe.faces, labels, reference=probe.baseline_vertices,
                                     title=title, warnings=[bool(group.loc[name, "geometry_warning"]) for name in names])
        stem = f"{case.split()[0].lower()}_{path}_{PLOT_SIZE_MODE}"
        fig.write_html(OUT / f"{stem}.html", include_plotlyjs=True)
        display(fig)
        static = plot_decoding_sequence_static(vertices, probe.faces, labels, title=title)
        static.savefig(OUT / f"{stem}.png", dpi=150)
        plt.close(static)

## Inspect opening errors and their effect on the interior

The next figures show the largest tested error in each sign, plus the unperturbed baseline. The finer modes are shown by default because they are the more demanding check. Smooth errors are included in the numerical tests and response plot; add 3 to `PLOT_RIM_MODES` to display their mesh sequences. The response plot compares the resulting core geometric error with the baseline reconstruction error.

The error directions are controlled probes, not measured errors from a trained decoder. Successful tests here cannot guarantee arbitrary coefficient samples are valid, or predict the eventual encoder's invariance.

In [ ]:
for case in cases:
    probe = probes[case, PLOT_SIZE_MODE]
    group = results[(results.case == case) & (results.size_mode == PLOT_SIZE_MODE)].set_index("scenario")
    for max_mode in PLOT_RIM_MODES:
        names = [f"rim_m{max_mode}_x1.00_-1", "baseline", f"rim_m{max_mode}_x1.00_+1"]
        vertices = [probe.matrix @ coefficients[case, PLOT_SIZE_MODE, name] for name in names]
        amplitude = group.loc[names[0], "rim_rms_relative"]*100
        labels = [f"− error: {amplitude:.1f}% rim RMS", "Baseline", f"+ error: {amplitude:.1f}% rim RMS"]
        title = f"{case}: opening modes 1–{max_mode} — {PLOT_SIZE_MODE}"
        fig = plot_decoding_sequence(vertices, probe.faces, labels, reference=probe.baseline_vertices,
                                     title=title, warnings=[bool(group.loc[name, "geometry_warning"]) for name in names])
        stem = f"{case.split()[0].lower()}_rim_m{max_mode}_{PLOT_SIZE_MODE}"
        fig.write_html(OUT / f"{stem}.html", include_plotlyjs=True)
        display(fig)
        static = plot_decoding_sequence_static(vertices, probe.faces, labels, title=title)
        static.savefig(OUT / f"{stem}.png", dpi=150)
        plt.close(static)
figure = plot_decoding_response(results, size_mode=PLOT_SIZE_MODE)
figure.savefig(OUT / f"opening_error_response_{PLOT_SIZE_MODE}.png", dpi=160)
display(figure)
plt.close(figure)

## What this establishes, and what it leaves to the learning project

A useful outcome is that observed boundary variation and modest coefficient errors lead to modest, continuous changes in the complete mesh, without geometric warnings. It does not require identical coefficients for different cuts. The core error should be interpreted against the baseline reconstruction and the morphological differences of interest, not an arbitrary universal percentage.

The within/between comparison uses a fixed cap-grid metric and only 20 pilot baselines. It does not establish anatomical correspondence or guarantee separation in latent space. The later encoder should test agreement of boundary variants while retaining differences between crypts. All variants from an organoid must stay in the same training/test split.

The saved test coefficients have a basis ID and explicit case/size/scenario labels; they are generated validation surfaces, not observed meshes with inherited source poses or measurements. Neither the latent model, a batch training-data exporter, nor scalar fields are implemented here.

Large source-pose changes identify an alignment issue that must remain visible to the learning project. Small source-frame errors combined with larger errors in the cohort frame do not indicate an unstable inverse mapping; they show that representation alignment is part of the remaining robustness problem.

In [ ]:
settings = dict(basis_id=basis.identifier, max_degree=basis.max_degree, standard_grid=GRID, fine_grid=FINE_GRID,
    interpolation_fractions=INTERPOLATION_FRACTIONS, paths=paths, rim_modes=RIM_MODES,
    error_multipliers=ERROR_MULTIPLIERS, seed_rule="2026 + max_mode", size_modes=SIZE_MODES,
    alignment="frozen saved pilot cohort references; no realignment of generated outputs",
    incident_contraction=1e-7, triangle_pair_interval_tolerance=1e-12,
    cases={case: dict(source=data["metadata"], boundary=data["boundary_diagnostics"]) for case, data in cases.items()},
    baseline_comparison_selection=selected,
    alignment_sha256={mode: hashlib.sha256((ARTIFACTS / f"pilot_alignment_{mode}.npz").read_bytes()).hexdigest()
                      for mode in SIZE_MODES}, skipped=skipped)
(OUT / "settings.json").write_text(json.dumps(settings, indent=2))
save_basis(OUT / "basis.json", basis)
print("Saved reports, test coefficients, settings and figures to", OUT)
print("Decoded meshes checked:", len(results), "at the standard grid and", len(fine), "at the fine grid.")